<a href="https://colab.research.google.com/github/jothikah/jothikah-OS-Lab-expr/blob/main/OS_Expr15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile disk_scheduling.c
#include <stdio.h>
#include <stdlib.h>

void fcfs(int requests[], int n, int head) {
    int total = 0;

    printf("\n--- FCFS Disk Scheduling ---\n");
    printf("Seek Sequence: %d", head);

    for (int i = 0; i < n; i++) {
        total += abs(head - requests[i]);
        head = requests[i];
        printf(" -> %d", head);
    }

    printf("\nTotal Head Movement = %d\n", total);
}

void sstf(int requests[], int n, int head) {
    int visited[100] = {0};
    int total = 0;

    printf("\n--- SSTF Disk Scheduling ---\n");
    printf("Seek Sequence: %d", head);

    for (int count = 0; count < n; count++) {
        int min = 99999;
        int index = -1;

        for (int i = 0; i < n; i++) {
            if (!visited[i]) {
                int distance = abs(head - requests[i]);

                if (distance < min) {
                    min = distance;
                    index = i;
                }
            }
        }

        visited[index] = 1;
        total += min;
        head = requests[index];

        printf(" -> %d", head);
    }

    printf("\nTotal Head Movement = %d\n", total);
}

void scan(int requests[], int n, int head, int diskSize, int direction) {
    int total = 0;
    int arr[100];
    int size = n;

    for (int i = 0; i < n; i++)
        arr[i] = requests[i];

    // Add disk boundary
    arr[size++] = direction == 1 ? diskSize - 1 : 0;

    // Sort
    for (int i = 0; i < size - 1; i++) {
        for (int j = i + 1; j < size; j++) {
            if (arr[i] > arr[j]) {
                int temp = arr[i];
                arr[i] = arr[j];
                arr[j] = temp;
            }
        }
    }

    int pos = 0;

    for (int i = 0; i < size; i++) {
        if (arr[i] >= head) {
            pos = i;
            break;
        }
    }

    printf("\n--- SCAN Disk Scheduling ---\n");
    printf("Seek Sequence: %d", head);

    if (direction == 1) {
        // Move towards higher tracks
        for (int i = pos; i < size; i++) {
            total += abs(head - arr[i]);
            head = arr[i];

            if (arr[i] != diskSize - 1)
                printf(" -> %d", head);
        }

        // Reverse direction
        for (int i = pos - 1; i >= 0; i--) {
            total += abs(head - arr[i]);
            head = arr[i];
            printf(" -> %d", head);
        }
    } else {
        // Move towards lower tracks
        for (int i = pos - 1; i >= 0; i--) {
            total += abs(head - arr[i]);
            head = arr[i];

            if (arr[i] != 0)
                printf(" -> %d", head);
        }

        // Reverse direction
        for (int i = pos; i < size; i++) {
            total += abs(head - arr[i]);
            head = arr[i];
            printf(" -> %d", head);
        }
    }

    printf("\nTotal Head Movement = %d\n", total);
}

void cscan(int requests[], int n, int head, int diskSize) {
    int total = 0;
    int arr[100];
    int size = n;

    for (int i = 0; i < n; i++)
        arr[i] = requests[i];

    // Add both disk boundaries
    arr[size++] = 0;
    arr[size++] = diskSize - 1;

    // Sort
    for (int i = 0; i < size - 1; i++) {
        for (int j = i + 1; j < size; j++) {
            if (arr[i] > arr[j]) {
                int temp = arr[i];
                arr[i] = arr[j];
                arr[j] = temp;
            }
        }
    }

    int pos = 0;

    for (int i = 0; i < size; i++) {
        if (arr[i] >= head) {
            pos = i;
            break;
        }
    }

    printf("\n--- C-SCAN Disk Scheduling ---\n");
    printf("Seek Sequence: %d", head);

    // Move towards higher tracks
    for (int i = pos; i < size; i++) {
        total += abs(head - arr[i]);
        head = arr[i];

        if (arr[i] != diskSize - 1)
            printf(" -> %d", head);
    }

    // Jump to beginning
    total += diskSize - 1;
    head = 0;

    printf(" -> 0");

    // Continue towards higher tracks
    for (int i = 1; i < pos; i++) {
        total += abs(head - arr[i]);
        head = arr[i];
        printf(" -> %d", head);
    }

    printf("\nTotal Head Movement = %d\n", total);
}

int main() {
    int n, head, diskSize, direction;
    int requests[100];

    printf("DISK SCHEDULING ALGORITHMS\n");
    printf("==========================\n");

    printf("Enter number of disk requests: ");
    scanf("%d", &n);

    printf("Enter request queue:\n");
    for (int i = 0; i < n; i++)
        scanf("%d", &requests[i]);

    printf("Enter initial head position: ");
    scanf("%d", &head);

    printf("Enter disk size: ");
    scanf("%d", &diskSize);

    printf("\nDirection for SCAN:\n");
    printf("1. Towards higher tracks\n");
    printf("0. Towards lower tracks\n");
    printf("Enter direction: ");
    scanf("%d", &direction);

    fcfs(requests, n, head);
    sstf(requests, n, head);
    scan(requests, n, head, diskSize, direction);
    cscan(requests, n, head, diskSize);

    return 0;
}

Writing disk_scheduling.c


In [ ]:
!gcc disk_scheduling.c -o disk_scheduling
!./disk_scheduling

DISK SCHEDULING ALGORITHMS
Enter number of disk requests: 9
Enter request queue:
5
5


In [ ]:
%%writefile disk_scheduling.sh
#!/bin/bash

echo "DISK SCHEDULING ALGORITHMS"
echo "=========================="

read -p "Enter number of disk requests: " n

echo "Enter request queue:"
read -a requests

read -p "Enter initial head position: " head
read -p "Enter disk size: " diskSize

# ---------------- FCFS ----------------

current=$head
total=0

echo ""
echo "--- FCFS ---"
echo -n "Seek Sequence: $current"

for ((i=0; i<n; i++))
do
    distance=$((current - requests[i]))

    if [ $distance -lt 0 ]
    then
        distance=$((-distance))
    fi

    total=$((total + distance))
    current=${requests[$i]}

    echo -n " -> $current"
done

echo ""
echo "Total Head Movement = $total"


# ---------------- SSTF ----------------

declare -a visited
for ((i=0; i<n; i++))
do
    visited[$i]=0
done

current=$head
total=0

echo ""
echo "--- SSTF ---"
echo -n "Seek Sequence: $current"

for ((count=0; count<n; count++))
do
    min=999999
    index=-1

    for ((i=0; i<n; i++))
    do
        if [ ${visited[$i]} -eq 0 ]
        then
            distance=$((current - requests[i]))

            if [ $distance -lt 0 ]
            then
                distance=$((-distance))
            fi

            if [ $distance -lt $min ]
            then
                min=$distance
                index=$i
            fi
        fi
    done

    visited[$index]=1
    total=$((total + min))
    current=${requests[$index]}

    echo -n " -> $current"
done

echo ""
echo "Total Head Movement = $total"


# ---------------- SCAN ----------------

echo ""
echo "--- SCAN ---"

# Sort requests
sorted=($(printf "%s\n" "${requests[@]}" | sort -n))

current=$head
total=0

echo -n "Seek Sequence: $current"

# Move towards higher tracks
for value in "${sorted[@]}"
do
    if [ $value -ge $current ]
    then
        distance=$((value - current))
        total=$((total + distance))
        current=$value

        echo -n " -> $current"
    fi
done

# Move to end of disk
distance=$((diskSize - 1 - current))
total=$((total + distance))
current=$((diskSize - 1))

echo -n " -> $current"

# Reverse direction
for ((i=${#sorted[@]}-1; i>=0; i--))
do
    value=${sorted[$i]}

    if [ $value -lt $head ]
    then
        distance=$((current - value))
        total=$((total + distance))
        current=$value

        echo -n " -> $current"
    fi
done

echo ""
echo "Total Head Movement = $total"


# ---------------- C-SCAN ----------------

echo ""
echo "--- C-SCAN ---"

current=$head
total=0

echo -n "Seek Sequence: $current"

# Move towards higher tracks
for value in "${sorted[@]}"
do
    if [ $value -ge $current ]
    then
        distance=$((value - current))
        total=$((total + distance))
        current=$value

        echo -n " -> $current"
    fi
done

# Move to end
distance=$((diskSize - 1 - current))
total=$((total + distance))
current=$((diskSize - 1))

echo -n " -> $current"

# Jump to beginning
total=$((total + diskSize - 1))
current=0

echo -n " -> $current"

# Continue from beginning
for value in "${sorted[@]}"
do
    if [ $value -lt $head ]
    then
        distance=$((value - current))
        total=$((total + distance))
        current=$value

        echo -n " -> $current"
    fi
done

echo ""
echo "Total Head Movement = $total"

In [ ]:
!chmod +x disk_scheduling.sh
!./disk_scheduling.sh